<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        🌳 00. Introducción a los Árboles de Decisión
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/00_Introduccion_Decision_Trees_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 00 de Árboles de Decisión. Si el cuaderno principal te habló de "segmentar el espacio de predictores en hiper-rectángulos" y sentiste que iba muy rápido, aquí vamos con calma: con analogías cotidianas y un mini-ejemplo que puedes ejecutar tú mismo, sin depender de ningún archivo externo.

No necesitas saber estadística avanzada — solo haber visto los módulos anteriores del curso (en especial el de Clasificación).

Al terminar podrás explicar, con tus propias palabras:
1. Qué es un árbol de decisión y por qué se parece tanto a un juego de adivinanzas.
2. Qué es la raíz (*root*), un nodo interno y una hoja (*leaf*).
3. Cómo entrenar un árbol sencillo con Scikit-Learn y "leer" sus reglas.
4. Cómo dibujar el árbol con `plot_tree` y entender lo que muestra cada caja.

---
## 1. Un árbol de decisión es el juego de "veinte preguntas" 🕵️

¿Jugaste alguna vez a "Veinte Preguntas"? Alguien piensa en algo (un animal, un objeto) y tú tienes que adivinarlo haciendo preguntas de sí/no: *"¿Es más grande que un perro?"*, *"¿Vive en el agua?"*... Cada respuesta descarta un montón de posibilidades y te acerca a la respuesta final.

Un **árbol de decisión** hace exactamente eso, pero con datos: en vez de "¿vive en el agua?", pregunta cosas como *"¿la edad es menor a 50 años?"* o *"¿el colesterol es mayor a 240?"*. Cada pregunta divide el grupo de datos en dos partes más "puras" (más parecidas entre sí), hasta que llega a una respuesta final.

> 📌 **Para recordar:** un árbol de decisión es una cadena de preguntas de sí/no, elegidas automáticamente por el algoritmo, que terminan en una respuesta (una predicción).

---
## 2. Anatomía de un árbol: raíz, ramas y hojas 🌿

Así como un árbol de verdad tiene raíz, tronco, ramas y hojas, un árbol de decisión tiene tres piezas:

```
                     ┌─────────────────────────┐
                     │   NODO RAÍZ (Root)      │  ◄── La primera pregunta,
                     │   ¿horas_estudio <= 4?  │      la más "poderosa" de todas
                     └────────────┬────────────┘
                                  │
                  ┌───────────────┴───────────────┐
                  ▼ Sí                            ▼ No
        ┌───────────────────┐            ┌────────────────────┐
        │   NODO INTERNO     │            │   HOJA: "Aprueba"  │  ◄── Ya no se
        │  ¿horas_sueno<=5?  │            └────────────────────┘      pregunta más:
        └─────────┬──────────┘                                        aquí vive la
                   │                                                  predicción final
        ┌──────────┴──────────┐
        ▼ Sí                 ▼ No
┌───────────────┐   ┌───────────────┐
│ HOJA: "Reprueba"│   │ HOJA: "Aprueba"│
└───────────────┘   └───────────────┘
```

1. **Nodo raíz (*root*):** la primera pregunta, la que el algoritmo considera más útil para separar los datos. Contiene el 100% de las observaciones antes de preguntar nada.
2. **Nodos internos:** preguntas intermedias que siguen refinando los grupos ("¿esta variable es mayor o menor que tal valor?").
3. **Hojas (*leaves*):** el final del camino. Ya no hay más preguntas — aquí el árbol entrega su predicción (por ejemplo, "Aprueba" o "Reprueba").

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("Entorno listo para construir nuestro primer árbol de decisión.")

---
## 3. Creamos nuestro propio mini-set de datos 🧪

En lugar de descargar un archivo externo, vamos a **inventar** (de forma controlada) un grupito de estudiantes ficticios con dos datos sencillos:

- `horas_estudio`: cuántas horas estudiaron para un examen.
- `horas_sueno`: cuántas horas durmieron la noche anterior.

Y queremos predecir: `aprobo` (1 si aprobó el examen, 0 si no). La regla "secreta" que usamos para generar los datos es simple (estudiar poco y dormir poco es mala combinación), pero le añadimos un poco de ruido aleatorio para que se parezca a la vida real.

In [ ]:
# Generamos 40 estudiantes ficticios
n = 40
horas_estudio = np.round(np.random.uniform(0, 9, n), 1)
horas_sueno = np.round(np.random.uniform(3, 9, n), 1)

# Regla "secreta" con algo de ruido: aprueba si estudió y durmió razonablemente
puntaje = 0.6 * horas_estudio + 0.4 * horas_sueno + np.random.normal(0, 1.2, n)
aprobo = (puntaje > 5.5).astype(int)

estudiantes = pd.DataFrame({
    'horas_estudio': horas_estudio,
    'horas_sueno': horas_sueno,
    'aprobo': aprobo
})

estudiantes.head(8)

### 🤔 ¿Qué acaba de pasar?

- Creamos 40 "estudiantes" con dos características (`horas_estudio` y `horas_sueno`) usando números aleatorios dentro de rangos razonables.
- Inventamos una regla oculta ("estudiar y dormir bien ayuda a aprobar") y le sumamos ruido aleatorio, para que los datos no sean perfectos ni 100% predecibles — igual que en la vida real.
- El resultado es una tabla (`DataFrame`) con tres columnas: dos características de entrada y una columna objetivo (`aprobo`) que es justo lo que le pediremos al árbol que aprenda a predecir.

---
## 4. Entrenando nuestro primer árbol con Scikit-Learn ⚡

Ya tenemos datos. Ahora le pedimos a `DecisionTreeClassifier` que **encuentre solo** las mejores preguntas para separar a los estudiantes que aprueban de los que no. Limitamos la profundidad (`max_depth=2`) para que el árbol sea pequeño y fácil de leer — como pedirle que juegue "veinte preguntas" pero usando máximo 2 preguntas.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

X = estudiantes[['horas_estudio', 'horas_sueno']]
y = estudiantes['aprobo']

arbolito = DecisionTreeClassifier(max_depth=2, random_state=42)
arbolito.fit(X, y)

print(f"Exactitud del árbol sobre estos mismos datos: {arbolito.score(X, y) * 100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

- `X` son las "preguntas disponibles" (las columnas que el árbol puede usar) y `y` es lo que queremos predecir.
- `DecisionTreeClassifier(max_depth=2, ...)` crea un árbol vacío y `.fit(X, y)` es el momento en el que el algoritmo **prueba muchísimas preguntas posibles** (¿`horas_estudio` mayor a 3? ¿a 4.5? ¿a 6?...) y se queda con las que mejor separan a los que aprueban de los que no.
- `max_depth=2` es como decirle "solo tienes permiso de hacer, como máximo, dos preguntas seguidas en cualquier camino" — así el árbol queda pequeño y entendible.
- `.score(X, y)` nos dice qué porcentaje de estudiantes clasificó correctamente el árbol ya entrenado.

In [ ]:
plt.figure(figsize=(11, 5.5), dpi=110)
plot_tree(
    arbolito,
    feature_names=['horas_estudio', 'horas_sueno'],
    class_names=['Reprueba', 'Aprueba'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Nuestro primer árbol de decisión (profundidad = 2)", fontweight='bold')
plt.show()

### 🤔 ¿Qué acaba de pasar?

`plot_tree` dibuja el árbol completo como un diagrama de cajas:

- La **caja de hasta arriba** es el **nodo raíz**: la pregunta que el algoritmo decidió que era la más útil de todas (probablemente algo como `horas_estudio <= valor`).
- Las **cajas intermedias** son **nodos internos**: siguen preguntando para refinar el grupo.
- Las **cajas de más abajo, sin flechas saliendo de ellas**, son las **hojas**: ahí ya no se pregunta nada más, y el color (más naranja o más azul, según la paleta que uses) te indica hacia qué clase se inclina esa hoja. El texto `value = [...]` te dice cuántos estudiantes de cada clase cayeron en esa caja.

In [ ]:
print("REGLAS DEL ÁRBOL EN TEXTO:")
print("=" * 55)
print(export_text(arbolito, feature_names=['horas_estudio', 'horas_sueno']))

### 🤔 ¿Qué acaba de pasar?

`export_text` traduce el mismo árbol del dibujo anterior a un formato de texto tipo "si... entonces...". Es exactamente el mismo modelo, solo que en vez de cajas y flechas, se lee como una receta de cocina: sigue las condiciones de arriba hacia abajo hasta llegar a un renglón que dice `class:`, y esa es la predicción final para ese camino.

---
## 5. ¿Por qué me debería importar todo esto? 🧭

Porque los árboles de decisión son la base de casi todos los modelos "estrella" que verás más adelante en este módulo: **Random Forests**, **Bagging**, **Boosting**... todos ellos son, en el fondo, *combinaciones de muchos árboles como el que acabas de entrenar*. Entender bien esta pieza pequeña (raíz, nodo interno, hoja, `plot_tree`) te va a servir para entender modelos mucho más potentes sin perderte.

---
##### 🛠️ Práctica 1: Lee tu árbol por dentro y úsalo

Ya tienes entrenado a `arbolito`. Es como un guardia con un cuestionario corto: veamos cuántas casillas tiene y probémoslo con un estudiante nuevo.

1. Guarda en `profundidad` cuántas preguntas seguidas hace como máximo (`get_depth()`) y en `n_hojas` cuántas hojas tiene (`get_n_leaves()`).
2. Crea una tabla de una sola fila llamada `nuevo` con un estudiante que estudió `8.0` horas y durmió `7.0` horas.
3. Pídele a `arbolito` su predicción con `predict(nuevo)[0]` y guárdala en `prediccion` (1 = aprueba, 0 = reprueba).

In [ ]:
# Escribe tu código aquí

# profundidad = arbolito.get_depth()
# n_hojas = arbolito.get_n_leaves()
# nuevo = pd.DataFrame({'horas_estudio': [8.0], 'horas_sueno': [...]})
# prediccion = arbolito.predict(...)[0]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Tamaño del árbol: preguntas seguidas y hojas finales
profundidad = arbolito.get_depth()
n_hojas = arbolito.get_n_leaves()

# 2) Un estudiante nuevo (una fila, con las mismas columnas que usó el árbol)
nuevo = pd.DataFrame({'horas_estudio': [8.0], 'horas_sueno': [7.0]})

# 3) Predicción del árbol
prediccion = arbolito.predict(nuevo)[0]

print(f"Profundidad: {profundidad} | Hojas: {n_hojas}")
print("Predicción:", "Aprueba" if prediccion == 1 else "Reprueba")

assert profundidad <= 2
assert n_hojas <= 4
assert prediccion == 1
```
</details>

---

---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Árbol de decisión | Una cadena de preguntas de sí/no, como el juego de "veinte preguntas", que termina en una predicción. |
| Nodo raíz | La primera y más importante pregunta; contiene el 100% de los datos antes de dividir. |
| Nodo interno | Una pregunta intermedia que sigue refinando los grupos. |
| Hoja (*leaf*) | El final de un camino: ahí vive la predicción (clase mayoritaria del grupo). |
| `DecisionTreeClassifier` | La clase de Scikit-Learn que entrena el árbol, probando muchas preguntas y quedándose con las mejores. |
| `plot_tree` | Dibuja el árbol completo como cajas y flechas. |
| `export_text` | Traduce el mismo árbol a reglas de texto tipo "si... entonces...". |

➡️ **Siguiente paso:** en el cuaderno [01 - Criterios de División y Árboles de Clasificación (Para Dummies)](01_Criterios_Division_y_Arboles_Clasificacion_Dummies.ipynb) descubrirás **cómo elige el algoritmo cuál es la "mejor" pregunta** en cada nodo, usando los criterios de Gini y Entropía.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>